# E. 도메인 평가 에이전트

| | |
|---|---|
| **담당** | RAG (B와 동일 하이브리드 파이프라인) |
| **선행 노드** | B |
| **출력** | `domain_eval`, `domain_references` |

도메인 평가 논문 3편을 검색해서 `tech_research`(B의 결과, 재사용)와 함께 OnDevice AI 환경에서 두 기술을 평가한다.
작동점 A/B는 구조화 필드를 두 번 안 채우고, 대표 판정(`criteria`, 8값) + 달라질 때만 서술(`reversal_criteria`)로 받는다 — design doc 3-1절 참고.

이 노트북 끝에서 만든 함수는 `src/nodes_e.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import prompts
from src.ingest import build_domain_retriever, format_docs_for_prompt
from src.node_utils import summarize_tech_research
from src.schemas import DomainEval

## 1. RAG 리트리버 준비

`data/papers/`에 `ondevice_eval.pdf`, `elib_mbu.pdf`, `llm_in_flash.pdf` 세 개가 있어야 한다.

In [ ]:
domain_retriever = build_domain_retriever()
print("리트리버 준비 완료")

## 2. 프롬프트 확인

메모리예산/정확도/지연/전력발열 각 판정 값에 카테고리뿐 아니라 실제 수치(GB/%p/ms/W)를 괄호로 같이 적으라는 지시가 들어있는지 확인.

In [ ]:
print(prompts.DOMAIN_EVAL_PROMPT)

## 3. 노드 함수 정의

In [ ]:
def make_node_e(llm, domain_retriever):
    """E. 도메인 평가. tech_research는 재사용(재검색 안 함), 도메인 논문만 새로 검색."""
    structured_llm = llm.with_structured_output(DomainEval)

    def node_e_domain_eval(state):
        tech_context = summarize_tech_research(state.get("tech_research", {}))
        queries = [
            "on-device LLM memory budget quantization accuracy tradeoff",
            "edge device memory bandwidth utilization inference latency",
            "flash memory offloading limited DRAM inference",
        ]
        all_docs = []
        for q in queries:
            all_docs.extend(domain_retriever.invoke(q))
        context = format_docs_for_prompt(all_docs, max_docs=10)

        prompt = prompts.DOMAIN_EVAL_PROMPT.format(
            tech_context=tech_context, context=context
        )
        result = structured_llm.invoke(prompt)
        refs = [
            {"source": d.metadata.get("source_name", "unknown"), "detail": d.page_content[:200]}
            for d in all_docs
        ]
        return {"domain_eval": result.model_dump(), "domain_references": refs}

    return node_e_domain_eval

## 4. 배선 테스트 — API 키 없이

In [ ]:
from src.schemas import DomainCriteria, TechStatus
from langchain_core.documents import Document

class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeLLM:
    def with_structured_output(self, schema_cls):
        dc = DomainCriteria(
            memory_budget=TechStatus(turboquant="들어간다 (1.2GB)", infinigen="넘는다 (1.8GB, 초과 0.3GB)"),
            accuracy=TechStatus(turboquant="나눠 보고 (-2%p)", infinigen="뭉쳐 보고 (-4%p)"),
            latency=TechStatus(turboquant="이 조건 실측 (45ms)", infinigen="다른 조건 실측 (배치8, 30ms)"),
            power_thermal=TechStatus(turboquant="실측 있음 (3.2W)", infinigen="근거 없음"),
        )
        return FakeStructuredLLM(DomainEval(
            criteria=dc, reversal_detected=True, reversal_criteria="메모리 예산 (가짜)",
            label="조건 의존", notes="가짜 근거",
        ))

class FakeRetriever:
    def invoke(self, query):
        return [Document(page_content=f"가짜 문서 for '{query}'", metadata={"source_name": "fake.pdf"})]

sample_state = {"tech_research": {"TurboQuant": {"overview": "가짜 개요"}}}

node_e = make_node_e(FakeLLM(), FakeRetriever())
result = node_e(sample_state)
assert result["domain_eval"]["criteria"]["memory_budget"]["infinigen"] == "넘는다 (1.8GB, 초과 0.3GB)"
assert len(result["domain_references"]) == 3  # RAG 3쿼리
print("배선 OK")
print(result["domain_eval"])

## 5. 실제 LLM 테스트

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY"):
    from langchain.chat_models import init_chat_model
    from src import config

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    node_e_real = make_node_e(real_llm, domain_retriever)
    print(node_e_real(sample_state)["domain_eval"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 6. 파일로 저장

In [ ]:
import inspect

TARGET = "../src/nodes_e.py"

# 심볼 유실 감지. 아래 parts 목록은 하드코딩이라, 누가 src/nodes_e.py 을 직접 고쳐
# 함수·상수를 더해 놓으면 저장하는 순간 그게 조용히 사라진다(2026-09-22 실제 발생).
# 사라진 이름이 있으면 여기서 알린다 - 에러가 안 나서 안 보이는 게 진짜 위험이다.
def _symbols(path):
    import ast, os
    if not os.path.exists(path):
        return set()
    out = set()
    for n in ast.parse(open(path, encoding="utf-8").read()).body:
        if isinstance(n, ast.FunctionDef):
            out.add(n.name)
        elif isinstance(n, ast.Assign):
            out |= {t.id for t in n.targets if isinstance(t, ast.Name)}
    return out


def _src(obj):
    """getsource 결과의 꼬리 개행을 없앤다. Jupyter 는 셀 끝에 개행을 붙이고
    스크립트 실행은 안 붙여서, 그대로 쓰면 환경마다 결과 파일이 달라진다."""
    return inspect.getsource(obj).rstrip("\n")


_before = _symbols(TARGET)

q3 = chr(34) * 3
HEADER = (
    q3 + "E. 도메인 평가 노드 - 03_agent_E_domain.ipynb에서 생성됨.\n"
    "이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n"
    "from src import prompts\n"
    "from src.ingest import format_docs_for_prompt\n"
    "from src.node_utils import summarize_tech_research\n"
    "from src.schemas import DomainEval\n\n\n"
)

parts = [
    _src(make_node_e),
]

with open(TARGET, "w", encoding="utf-8") as f:
    f.write(HEADER + "\n\n\n".join(parts) + "\n")   # 정의 사이는 빈 줄 2개(PEP8)

_lost = sorted(_before - _symbols(TARGET))
if _lost:
    print(f"🔴 이번 저장으로 {TARGET} 에서 사라진 심볼: {_lost}")
    print("   노트북이 .py 보다 낡았다는 뜻이다. git diff 로 확인하고,")
    print("   의도한 삭제가 아니면 git checkout 으로 되돌린 뒤 노트북부터 맞출 것.")
else:
    print(f"{TARGET} 저장 완료 (심볼 유실 없음)")